# Proyecto: Predicción de Episodios de Mala Calidad del Aire en México
**Objetivo:** Desarrollar un modelo predictivo integrando datos de contaminantes atmosféricos (SINAICA), clima (CONAGUA) y factores urbanos (INEGI).

## 1. Carga y Exploración de Datos (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data/data_processed.csv')
df.head()

### Valores Faltantes y Limpieza Inicial

In [ ]:
print(df.isnull().sum())
# En el script de preprocesamiento aplicamos bfill (backward fill) para tratar los valores faltantes.
df.bfill(inplace=True)

### Matriz de Correlación

In [ ]:
plt.figure(figsize=(10,6))
cols_num = ['PM2.5', 'PM10', 'O3', 'Temperatura', 'Humedad', 'Mala_Calidad_Aire']
sns.heatmap(df[cols_num].corr(), annot=True, cmap='coolwarm')
plt.title('Correlación de Variables')
plt.show()

## 2. Ingeniería de Características
Creamos promedios móviles (rezagos de 3 días) para simular el comportamiento acumulado del aire.

In [ ]:
for col in ['PM2.5', 'PM10', 'Temperatura']:
    df[f'{col}_ma3'] = df.groupby('Estacion')[col].transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
df.bfill(inplace=True)

## 3. Modelado Predictivo
Entrenaremos los 4 modelos requeridos: Regresión Logística, Random Forest, XGBoost y Red Neuronal.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

X = df.drop(columns=['Fecha', 'Estacion', 'Region', 'PM2.5', 'PM10', 'O3', 'NO2', 'Mala_Calidad_Aire'])
y = df['Mala_Calidad_Aire']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

### Entrenamiento y Evaluación

In [ ]:
modelos = {
    'Logistica': LogisticRegression(),
    'Random Forest': RandomForestClassifier(n_estimators=50),
    'XGBoost': XGBClassifier(eval_metric='logloss'),
    'MLP (Red Neuronal)': MLPClassifier(hidden_layer_sizes=(50,), max_iter=300)
}

for nombre, mod in modelos.items():
    mod.fit(X_train_s, y_train)
    y_pred = mod.predict(X_test_s)
    auc = roc_auc_score(y_test, mod.predict_proba(X_test_s)[:, 1]) if hasattr(mod, 'predict_proba') else 'N/A'
    print(f'\n=== {nombre} ===')
    print(f'AUC-ROC: {auc}')
    print(classification_report(y_test, y_pred))
